Importing Basic Libraries

In [ ]:
import pandas as pd
import numpy as np

Importing Regression Funcitons

In [ ]:
from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
import lightgbm as lgb

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

Importing Evaluator Functions

In [ ]:
!pip install sktime

In [ ]:
from sklearn.metrics import r2_score

from sklearn.metrics import mean_absolute_percentage_error
from sktime.performance_metrics.forecasting import median_absolute_percentage_error

Importing Classes and Functions from Alternate Files

In [ ]:
from model_classes import W4_Regression, W5_Regs, W7_Boosting

In [ ]:
# Preset file

filename = "CRMLS_0625-0626_enriched.csv"
end_mnth = 6

In [ ]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [ ]:
enr_df = load_df()

In [ ]:
# Preset Values

main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols


cols = enr_df.columns.to_list()
new_col_strt = [i for i in range(len(cols)) if cols[i] == "SaleMonth"]
new_cols = cols[(new_col_strt[0]+1):len(cols)]
new_cols = [i for i in new_cols if i != "DistrictName"]

crit_cols = totals+new_cols
log_cols = main_cols+new_cols


target = "ClosePrice"

In [ ]:
class W8_Eval():

  def __init__(self,df):
    self.df = df


  def mape_eval(self, y_test, y_pred):
    mape = mean_absolute_percentage_error(y_test, y_pred)       # Computes the mean absolute percentage error of y_test and the predicted y
    return mape


  def mdape_eval(self, y_test, y_pred):
    mdape = median_absolute_percentage_error(y_test, y_pred)       # Computes the median absolute percentage error of y_test and the predicted y
    return mdape

In [ ]:
base = W4_Regression(enr_df)
comp = W5_Regs(enr_df)
boost = W7_Boosting(enr_df)
Week8 = W8_Eval(enr_df)

train, test = base.test_train_split()


log_df = base.log_transform()

bs = W4_Regression(log_df)
cp = W5_Regs(log_df)
bst = W7_Boosting(log_df)
Wk8 = W8_Eval(log_df)

tr, te = bs.test_train_split()

# **Evaluation:  *MAPE, MdAPE***

 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

**Linear Regression**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LRmape, LRmape = comp.shrt_main(base.LinReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmape, logLRmape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LRmdape, LRmdape = comp.shrt_main(base.LinReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmdape, logLRmdape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

**Decision Tree Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_DTRmape, DTRmape = comp.shrt_main(comp.TreeReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmape, logDTRmape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_DTRmdape, DTRmdape = comp.shrt_main(comp.TreeReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmdape, logDTRmdape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

**Random Forest Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_RFRmape, RFRmape = comp.shrt_main(comp.ForestReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmape, logRFRmape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_RFRmdape, RFRmdape = comp.shrt_main(comp.ForestReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmdape, logRFRmdape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

In [ ]:
adv_results = main()

**XGBoost**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_XGBmape, XGBmape = comp.shrt_main(boost.XGB, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                    dep=adv_results["max_depth"].iloc[0], l_rate=adv_results["learning_rate"].iloc[0], est=adv_results["n_estimators"].iloc[0])

print("Log Transform")
c_logXGBmape, logXGBmape = cp.shrt_main(bst.XGB, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                        dep=adv_results["max_depth"].iloc[1], l_rate=adv_results["learning_rate"].iloc[1], est=adv_results["n_estimators"].iloc[1])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_XGBmdape, XGBmdape = comp.shrt_main(boost.XGB, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                      dep=adv_results["max_depth"].iloc[0], l_rate=adv_results["learning_rate"].iloc[0], est=adv_results["n_estimators"].iloc[0])

print("Log Transform")
c_logXGBmdape, logXGBmdape = cp.shrt_main(bst.XGB, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                          dep=adv_results["max_depth"].iloc[1], l_rate=adv_results["learning_rate"].iloc[1], est=adv_results["n_estimators"].iloc[1])

**Gradient Boosting Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_GBRmape, GBRmape = comp.shrt_main(boost.GBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                    dep=adv_results["max_depth"].iloc[2], l_rate=adv_results["learning_rate"].iloc[2], est=adv_results["n_estimators"].iloc[2])

print("Log Transform")
c_logGBRmape, logGBRmape = cp.shrt_main(bst.GBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                        dep=adv_results["max_depth"].iloc[3], l_rate=adv_results["learning_rate"].iloc[3], est=adv_results["n_estimators"].iloc[3])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_GBRdmape, GBRmdape = comp.shrt_main(boost.GBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                      dep=adv_results["max_depth"].iloc[2], l_rate=adv_results["learning_rate"].iloc[2], est=adv_results["n_estimators"].iloc[2])

print("Log Transform")
c_logGBRdmape, logGBRmdape = cp.shrt_main(bst.GBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                          dep=adv_results["max_depth"].iloc[3], l_rate=adv_results["learning_rate"].iloc[3], est=adv_results["n_estimators"].iloc[3])

**LightGBM**

In [ ]:
# MAPE

In [ ]:
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LGBMmape, LGBMmape = comp.shrt_main(boost.L_GBM, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                      dep=adv_results["max_depth"].iloc[4], l_rate=adv_results["learning_rate"].iloc[4], est=adv_results["n_estimators"].iloc[4])

In [ ]:
print("Log Transform")
c_logLGBMmape, logLGBMmape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                          dep=adv_results["max_depth"].iloc[5], l_rate=adv_results["learning_rate"].iloc[5], est=adv_results["n_estimators"].iloc[5])

In [ ]:
# MdAPE

In [ ]:
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LGBMmdape, LGBMmdape = comp.shrt_main(boost.L_GBM, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                        dep=adv_results["max_depth"].iloc[4], l_rate=adv_results["learning_rate"].iloc[4], est=adv_results["n_estimators"].iloc[4])

In [ ]:
print("Log Transform")
c_logLGBMmdape, logLGBMmdape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                            dep=adv_results["max_depth"].iloc[5], l_rate=adv_results["learning_rate"].iloc[5], est=adv_results["n_estimators"].iloc[5])

***Mean Absolute Percentage Error***

*Non-Transform*

* ***0.0181***

*Log Transform*

* ***0.0007***

***Median Absolute Percentage Error***

*Non-Transform*

* ***0.01***

*Log Transform*

* ***0.0007***

**Histogram-based Gradient Boosting Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_HGBRmape, HGBRmape = comp.shrt_main(boost.HGBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                      dep=adv_results["max_depth"].iloc[6], l_rate=adv_results["learning_rate"].iloc[6])

print("Log Transform")
c_logHGBRmape, logHGBRmape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                          dep=adv_results["max_depth"].iloc[7], l_rate=adv_results["learning_rate"].iloc[7])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_HGBRmdape, HGBRmdape = comp.shrt_main(boost.HGBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                        dep=adv_results["max_depth"].iloc[6], l_rate=adv_results["learning_rate"].iloc[6])

print("Log Transform")
c_logHGBRmdape, logHGBRmdape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                            dep=adv_results["max_depth"].iloc[7], l_rate=adv_results["learning_rate"].iloc[7])

***Summary***

---

 Highest MAPE:
   * Non-Transform: **.0013**
     * **Random Forest**
   * Log Transform: **.0001**
     * **Random Forest / Decision Tree**

---

 Highest MdAPE: **Decision Tree**
   * Non-Transform: **0.0**
   * Log Transform: **0.0**

# **Price Bands**

In [ ]:
# Defines column "PriceRank" with .qcut from 'Low' to 'High'
enr_df["PriceRank"] = pd.qcut(enr_df["ClosePrice"], q=5, labels=["Low", "Low-Medium", "Medium", "Medium-High", "High"])

**Bin Values**
 * ***Q1: High***
   * `$1,610,000 - $110,000,000`
   * (1610000.0, 110000000.0]
 * ***Q2: Medium-High***
   * `$1,086,000 - $1,610,000`
   * (1086000.0, 1610000.0]
 * ***Q3: Medium***
   * `$791,000 - $1,086,000`
   * (791000.0, 1086000.0]
 * ***Q4: Low-Medium***
   * `$574,000 - $791,000`
   * (574000.0, 791000.0]
 * ***Q5: Low***
   * `$26,000 - $574,000`
   * (25999.999, 574000.0]

In [ ]:
# Bin Values
(sorted(pd.unique(pd.qcut(enr_df["ClosePrice"], q=5)), reverse=True))

In [ ]:
def quant_eval(df, q_val, modl_typ, feats):

  qdf = df[df["PriceRank"] == q_val]

  wk4, wk5 = W4_Regression(qdf), W5_Regs(qdf)
  wk7, wk8 = W7_Boosting(qdf), W8_Eval(qdf)
  train, test = base.test_train_split()

  qlog_df = base.log_transform()
  w4, w5 = W4_Regression(qlog_df), W5_Regs(qlog_df)
  w7, w8 = W7_Boosting(qlog_df), W8_Eval(qlog_df)
  tr, te = bs.test_train_split()


  if modl_typ == "Linear Regression":              # Linear Regression
    r2 = wk5.shrt_main(wk4.LinReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk4.LinReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk4.LinReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w4.LinReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w4.LinReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w4.LinReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Decision Tree Regressor":             # Decision Tree Regressor
    r2 = wk5.shrt_main(wk5.TreeReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk5.TreeReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk5.TreeReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w5.TreeReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w5.TreeReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w5.TreeReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Random Forest Regressor":             # Random Forest Regressor
    r2 = wk5.shrt_main(wk5.ForestReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk5.ForestReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk5.ForestReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w5.ForestReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w5.ForestReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w5.ForestReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "XGBoost":             # XGBoost

    r2 = wk5.shrt_main(wk7.XGB(dep=7, l_rate=0.2, est=200), train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.XGB(dep=7, l_rate=0.2, est=200), train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk7.XGB(dep=7, l_rate=0.2, est=200), train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w7.XGB(dep=7, l_rate=0.2, est=200), tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.XGB(dep=7, l_rate=0.2, est=200), tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w7.XGB(dep=7, l_rate=0.2, est=200), tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Gradient Boosting Regressor":             # Gradient Boosting Regressor

    r2 = wk5.shrt_main(wk7.GBR(dep=3, l_rate=0.25, est=200), train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.GBR(dep=3, l_rate=0.25, est=200), train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk7.GBR(dep=3, l_rate=0.25, est=200), train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w7.GBR(dep=3, l_rate=0.25, est=200), tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.GBR(dep=3, l_rate=0.25, est=200), tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w7.GBR(dep=3, l_rate=0.25, est=200), tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "LightGBM":             # LightGBM

    r2 = wk5.shrt_main(wk7.L_GBM(dep=5, l_rate=0.5, est=200), train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.L_GBM(dep=5, l_rate=0.5, est=200), train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk7.L_GBM(dep=5, l_rate=0.5, est=200), train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w7.L_GBM(dep=5, l_rate=0.5, est=200), tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.L_GBM(dep=5, l_rate=0.5, est=200), tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w7.L_GBM(dep=5, l_rate=0.5, est=200), tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Histogram-based Gradient Boosting Regressor":             # Histogram-based Gradient Boosting Regressor

    r2 = wk5.shrt_main(wk7.HGBR(dep=5, l_rate=0.75), train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.HGBR(dep=5, l_rate=0.75), train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk7.HGBR(dep=5, l_rate=0.75), train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w7.HGBR(dep=5, l_rate=0.75), tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.HGBR(dep=5, l_rate=0.75), tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w7.HGBR(dep=5, l_rate=0.75), tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)

  new_df = pd.DataFrame({"Model": [modl_typ]*2, "LogForm": [False,True], "R2 Score": [r2,lg_r2], "MAPE": [mape,lg_mape], "MdAPE": [mdape,lg_mdape]})
  return new_df

# **Metrics DataFrame: *R2, MAPE, MdAPE***

 * R2 Score (R2)
 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

In [ ]:
modl = ["LinearRegression"]*4 + ["Decision Tree Regressor"]*4 + ["Random Forest Regressor"]*4 + [
    "XGBoost"]*4 + ["Gradient Boosting Regressor"]*4 + ["LightGBM"]*4 + ["Histogram-based Gradient Boosting Regressor"]*4

lg = [False, True]*14

typ = ["MAPE", "MAPE", "MdAPE", "MdAPE"]*7

cols = [c_LRmape, c_logLRmape, c_LRmape, c_logLRmape] + [c_DTRmape, c_logDTRmape, c_DTRmdape, c_logDTRmdape] + [
    c_RFRmape, c_logRFRmape, c_RFRmdape, c_logRFRmdape] + [c_XGBmape, c_logXGBmape, c_XGBmdape, c_logXGBmdape] + [
        c_GBRmape, c_logGBRmape, c_GBRmdape, c_logGBRmdape] + [c_LGBMmape, c_logLGBMmape, c_LGBMmdape, c_logLGBMmdape] + [
        c_HGBRmape, c_logHGBRmape, c_HGBRmdape, c_logHGBRmdape]

scrs = [LRmape, logLRmape, LRmdape, logLRmdape] + [DTRmape, logDTRmape, DTRmdape, logDTRmdape] + [RFRmape, logRFRmape, RFRmdape, logRFRmdape] + [
    XGBmape, logXGBmape, XGBmdape, logXGBmdape] + [GBRmape, logGBRmape, GBRmdape, logGBRmdape] + [LGBMmape, logLGBMmape, LGBMmdape, logLGBMmdape] + [
        HGBRmape, logHGBRmape, HGBRmdape, logHGBRmdape]

In [ ]:
md = [None]*12 + [adv_results["max_depth"].iloc[0]]*2 + [adv_results["max_depth"].iloc[2]]*2 + [
    adv_results["max_depth"].iloc[4]]*2 + [adv_results["max_depth"].iloc[6]]*2

lr = [None]*12 + [adv_results["learning_rate"].iloc[0]]*2 + [adv_results["learning_rate"].iloc[2]]*2 + [
    adv_results["learning_rate"].iloc[4]]*2 + [adv_results["learning_rate"].iloc[6]]*2

ne = [None]*12 + [adv_results["n_estimators"].iloc[0]]*2 + [adv_results["n_estimators"].iloc[2]]*2 + [
    adv_results["n_estimators"].iloc[4]]*2 + [adv_results["n_estimators"].iloc[6]]*2

In [ ]:
mets = {"Model": modl, "LogForm": lg, "ScoreType": typ, "ScoreValue": scrs, "max_depth": md, "learning_rate": lr, "n_estimators": ne, "columns": cols}

met_df = pd.DataFrame(mets)
met_df = adv_results.concat(met_df)

In [ ]:
# Preset File

metrics_df = "metrics_summary.csv"

In [ ]:
def save_csv(df, file=metrics_df):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(file, index=False)

In [ ]:
save_csv()